
## **Univariate Time Series** vs **Multivariate Time Series**

---

### **Univariate Time Series**

* **Definition:**
  A time series that consists of observations of a **single variable** collected over time.

* **Example:**
  Daily **temperature** in a city over a year:
  `Temp(t1), Temp(t2), Temp(t3), ..., Temp(tn)`

* **Characteristics:**

  * Only one feature (dependent variable) is tracked.
  * Models like **ARIMA**, **Simple Exponential Smoothing**, **Prophet** (basic), etc., are used.
  * Easier to model and interpret.
  * Assumes past values of the same variable carry enough information to predict the future.

* **Use Cases:**

  * Stock price prediction (closing price only)
  * Forecasting electricity demand from past demand
  * Website traffic forecasting based on past traffic only

---

### **Multivariate Time Series**

* **Definition:**
  A time series that includes **multiple variables (features)** observed over time. These variables may be dependent or independent and can affect each other.

* **Example:**
  Predicting future **sales** using:

  * `Sales(t)`
  * `Marketing Spend(t)`
  * `Holiday(t)`
  * `Temperature(t)`

  i.e., `Y(t) = f(Sales(t−1), Marketing(t), Holiday(t), Weather(t))`

* **Characteristics:**

  * Contains more than one time-dependent variable.
  * Captures **relationships and interactions** between variables.
  * Requires more advanced models:

    * **Vector Autoregression (VAR)**
    * **LSTM/GRU with multivariate input**
    * **XGBoost/LightGBM with lag features**
    * **Facebook Prophet with regressors**

* **Use Cases:**

  * Sales forecasting using promotions, holidays, and economic indicators
  * Demand forecasting using weather, price, and past demand
  * Energy consumption prediction using temperature, time of day, and occupancy

---

### Key Differences

| Aspect            | Univariate                      | Multivariate                                  |
| ----------------- | ------------------------------- | --------------------------------------------- |
| Variables         | One                             | Multiple                                      |
| Complexity        | Simpler                         | More complex                                  |
| Relationships     | No inter-variable relationships | Considers inter-variable relationships        |
| Forecasting Power | May be limited                  | Usually higher with correct features          |
| Example           | Temperature over time           | Temperature, humidity, and pressure over time |

---


## Splitting Time Series Data

### `TimeSeriesSplit` (scikit-learn) — Detailed Explanation

- **What it is:** `TimeSeriesSplit` is a cross-validation iterator specifically for time series data. It preserves temporal order and avoids shuffling, ensuring the training data for each fold contains only observations that occur before the validation fold.

- **How it works:** For `n_splits = k`, `TimeSeriesSplit` produces k successive train/test splits where each test set is a contiguous block that follows its training set. The training set typically grows with each split (an expanding window), while test sets are sequential and non-overlapping.

- **Key parameters:**

  - `n_splits`: Number of folds/splits (required).
  - `max_train_size`: Optional cap on training set size (enables a rolling window).
  - `test_size`: Optional fixed size for each test fold (if provided).
  - `gap`: Number of samples to exclude between train and test to avoid leakage (introduced in newer sklearn versions).

- **Typical behavior (example):** For `n_samples=12` and `n_splits=3` you often see an expanding-window pattern like:

  - Split 1: train indices [0..2], test [3..5]
  - Split 2: train indices [0..5], test [6..8]
  - Split 3: train indices [0..8], test [9..11]

- **When to use:**

  - Use for time series model selection where preserving time order is essential and future leakage must be avoided.
  - Use with feature engineering that uses only past data (lags, rolling statistics, expanding features).
  - When concept drift is possible, consider `max_train_size` (rolling window) to limit older data influence.

- **Practical tips:**

  - Inspect indices produced by `.split()` to verify splits before training.
  - Use `gap` when labels are derived from future windows or when there is an operational delay between features and labels.
  - Aggregate fold metrics in a time-aware way (e.g., weighted by time or by fold relevance) rather than relying on i.i.d. CV assumptions.
  - For panel / multiple time series, apply splits per series or use specialized splitters that respect grouping (e.g., `GroupKFold` combined with time-aware logic).

- **When not to use:**

  - Do not use standard shuffled k-fold CV for time series unless you can safely assume i.i.d. data across time.

Below the notebook contains a short runnable example demonstrating `TimeSeriesSplit`. Review the printed indices to ensure splits match your expectations and adjust `n_splits`, `max_train_size`, `test_size`, or `gap` accordingly.

In [1]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

# Generate some sample time series data
X = np.arange(100).reshape(-1, 1)
y = np.sin(np.arange(100))

# Initialize TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

print("Using TimeSeriesSplit with n_splits=5:")

# Iterate through the splits and print the indices
for train_index, test_index in tscv.split(X):
    print(f"TRAIN indices: {train_index.tolist()}")
    print(f"TEST indices: {test_index.tolist()}\n")

# You can then use these indices to split your data
# X_train, X_test = X[train_index], X[test_index]
# y_train, y_test = y[train_index], y[test_index]

Using TimeSeriesSplit with n_splits=5:
TRAIN indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
TEST indices: [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]

TRAIN indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
TEST indices: [36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

TRAIN indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
TEST indices: [52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67]

TRAIN indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 6

# **Need to know - Time Series Analysis**

**Supervised and Unsupervised ML** already gives us a solid foundation. To **master Time Series Analysis** from both a **data science interview** and **real-world modeling** perspective, apart from the above topics we discussed so far, here’s a structured **topics you must know**.

---

## Phase 1: **Time Series Basics**

> Goal: Understand what makes time series unique (temporal dependencies)

* 📌 What is a time series?
* 📌 Types of time series: univariate vs multivariate
* 📌 Components of time series:

  * Trend
  * Seasonality
  * Cyclicity
  * Noise
* 📌 Time series vs supervised learning
* 📌 Stationarity (conceptual & statistical)

  * Augmented Dickey-Fuller (ADF) test
  * KPSS test
* 📌 Time-based train-test split

---

## Phase 2: **Classical Time Series Modeling**

> Goal: Learn statistical methods that are interview favorites and work well for short/structured datasets

* 📌 Differencing & transforming series
* 📌 ACF and PACF interpretation (AR vs MA vs ARMA)
* 📌 ARIMA (AutoRegressive Integrated Moving Average)

  * Choosing p, d, q
  * AIC/BIC tuning
* 📌 Seasonal ARIMA (SARIMA)

  * Adding seasonal orders (P, D, Q, s)
* 📌 Auto ARIMA (pmdarima)
* 📌 Residual diagnostics
* 📌 Forecasting using ARIMA/SARIMA

---

## Phase 3: **Time Series Features + Supervised Framing**

> Goal: Frame time series problems as supervised learning

* 📌 Lag features
* 📌 Rolling window (moving average, min/max, std)
* 📌 Expanding window stats
* 📌 Time-based features:

  * Day of week, month, holiday flag, etc.
* 📌 Framing with X and y: converting time series into supervised format
* 📌 Train/validation split using `TimeSeriesSplit`
* 📌 Walk-forward validation

---

## Phase 4: **Machine Learning for Time Series**

> Goal: Use ML models (XGBoost, RF, etc.) for time series forecasting

* 📌 XGBoost / LightGBM for time series
* 📌 Feature engineering recap (lags, trends, seasonality)
* 📌 Handling leakage (no peeking into the future)
* 📌 Hyperparameter tuning with time-based CV
* 📌 Quantile regression for forecasting intervals

---

## Phase 5: **Time Series Deep Learning**

> Goal: Go beyond traditional ML — handle complex, nonlinear, long sequences

* 📌 RNNs (basic concepts)
* 📌 LSTM / GRU networks
* 📌 Sequence-to-sequence models
* 📌 CNN for time series
* 📌 Temporal Fusion Transformers (advanced)
* 📌 Libraries: `TensorFlow`, `PyTorch`, `Darts`, `GluonTS`, `Kats`

---

## Phase 6: **Advanced & Real-World Applications**

> Goal: Be job-ready for interviews and deployments

* 📌 Hierarchical time series
* 📌 Multivariate forecasting
* 📌 Anomaly detection in time series
* 📌 Forecast accuracy metrics:

  * MAE, RMSE, MAPE, SMAPE, MASE, etc.
* 📌 Backtesting strategies
* 📌 Production-ready forecasting pipeline
* 📌 Model monitoring & data drift
* 📌 Tools: Prophet, Darts, Statsmodels, pmdarima

---

## Bonus: **Interview-Oriented Concepts**

> Goal: Crack interviews confidently

* 📌 Why ARIMA/SARIMA over XGBoost?
* 📌 How do you handle holidays/special events?
* 📌 What is leakage in time series modeling?
* 📌 How do you evaluate a model's performance?
* 📌 Describe a project where you used time series
* 📌 How would you design demand forecasting for 10,000 SKUs?
* 📌 How would you scale your model for millions of time series?

---

